# temp-setup-secrets — Configuração do Secret Scope

**Objetivo:** Criar o Databricks Secret Scope `capgemini_academy` e armazenar a chave da API exchangeratesapi.io de forma segura.

| Item | Valor |
|---|---|
| **Camada** | Setup / infraestrutura (executar uma vez) |
| **Entrada** | Token do Databricks e chave da API (via widgets, NÃO hardcoded) |
| **Saída** | Secret Scope `capgemini_academy` com a key `exchangerates_api_key` |
| **Ordem no pipeline** | Antes de `Day1-Ingestao-Exchange-Rates` |

> **🚨 ALERTA DE SEGURANÇA:**
> A versão original deste notebook continha:
> - **Token do Databricks hardcoded** (exposto em texto plano)
> - **Chave da API exchangeratesapi.io hardcoded** (exposta em texto plano)
> - **URL do workspace** hardcoded
>
> Esses valores foram **removidos** e substituídos por widgets. **Nunca** versione credenciais em código.
> Antes de executar, forneça os valores via os widgets abaixo.

## Widgets de configuração (sem hardcoded secrets)

Forneça os valores via widgets. O token e a chave da API **não** ficam salvos no notebook — são passados em runtime.

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
dbutils.widgets.text("databricks_token", "", "Token do Databricks (NÃO versionar)")
dbutils.widgets.text("api_key", "", "Exchange Rates API Key (NÃO versionar)")
dbutils.widgets.text("workspace_host", "https://dbc-2caf0d2f-861c.cloud.databricks.com", "URL do workspace Databricks")

catalog = dbutils.widgets.get("catalog")
token = dbutils.widgets.get("databricks_token")
api_key = dbutils.widgets.get("api_key")
host = dbutils.widgets.get("workspace_host")

# NOTA: Os valores acima sao lidos em runtime via widgets.
# Eles NAO sao persistidos no notebook e NAO serao commitados no Git.

## Criação do Secret Scope e armazenamento da chave

Usa a API REST do Databricks para criar o scope e guardar a chave. Substitui os valores hardcoded originais por variaveis lidas dos widgets.

In [0]:
import requests

headers = {"Authorization": f"Bearer {token}"}

# Cria o scope (a "gaveta" onde a chave da API vai morar)
r1 = requests.post(
    f"{host}/api/2.0/secrets/scopes/create",
    headers=headers,
    json={"scope": catalog}
)
print("Criar scope:", r1.status_code, r1.text)

# Guarda a chave da API dentro dessa gaveta
r2 = requests.post(
    f"{host}/api/2.0/secrets/put",
    headers=headers,
    json={
        "scope": catalog,
        "key": "exchangerates_api_key",
        "string_value": api_key
    }
)
print("Guardar chave:", r2.status_code, r2.text)